In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv("variants_new_20k_with_scores.csv")

drop_cols = [
    'Name', 'GeneSymbol', 'ClinicalSignificance', 'PhenotypeList', 
    'ReferenceAllele', 'AlternateAllele', 'ReferenceAlleleVCF', 'AlternateAlleleVCF'
]

X = df.drop(columns=drop_cols + ['target'])
y = df['target']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, 
                                    stratify = y, random_state = 42)

imputer = SimpleImputer(strategy='mean')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

In [18]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=1000,                 
    learning_rate=0.03, 
    max_depth=5, random_state=42,
    eval_metric = 'logloss'
)
xgb_clf.fit(X_train,y_train)
y_pred_xgb = xgb_clf.predict(X_test)
f1_xgb = f1_score(y_test, y_pred_xgb)
print("XGBoost: ", f1_xgb) 


cat_clf = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=7,
    random_seed=42,
    verbose=0
)
cat_clf.fit(X_train, y_train,)
y_pred_cat = cat_clf.predict(X_test)
f1_cat = f1_score(y_test, y_pred_cat)
print("CatBoost: ", f1_cat)


lgb_clf = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
lgb_clf.fit(X_train, y_train)
y_pred_lgb = lgb_clf.predict(X_test)
f1_lgb = f1_score(y_test, y_pred_lgb)
print("LighBoost: ", f1_lgb)


rf_clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf)
print("Random Forest: ", f1_rf)


mlp_clf = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    verbose=False
)
mlp_clf.fit(X_train, y_train)
y_pred_mlp = mlp_clf.predict(X_test)
f1_mlp = f1_score(y_test, y_pred_mlp)
print("Neural Network:" , f1_mlp)

XGBoost:  0.8904452226113057
CatBoost:  0.8926157697121402
LighBoost:  0.8874338957441451
Random Forest:  0.8890532544378699
Neural Network: 0.8690977092099111
